# W9 Exercise 2 - ADP, Q-learning, and Policy Search

This practice notebook applies three reinforcement learning methods from the Week 9 lecture to a small task.

The environment is intentionally close in size to the classic two-location examples, but it is not the vacuum-cleaner world used in the demo notebook.



## Environment: Package Delivery World

A robot moves between two locations: Store and Home.

The package status can be waiting, carrying, or delivered.

The state is:

$
s = (\text{location}, \text{package status})
$

The action set is:

$
A = \{\text{GoStore}, \text{GoHome}, \text{PickUp}, \text{Deliver}, \text{Wait}\}
$

Rewards:

- Valid PickUp at Store when the package is waiting: $+5$
- Valid Deliver at Home when carrying the package: $+20$
- Invalid pickup or delivery: $-3$
- Movement and waiting: $-1$
- Terminal delivered states: $0$

Movement succeeds with probability $0.90$; otherwise the robot stays in the same location.



In [1]:
from collections import Counter, defaultdict
from itertools import product
import random
import statistics

# Robot có thể đứng ở một trong hai vị trí.
LOCATIONS = ("Store", "Home")

# Trạng thái của gói hàng tóm tắt tiến độ nhiệm vụ.
PACKAGE_STATUSES = ("waiting", "carrying", "delivered")

# Đây là tất cả action có thể dùng ở mọi non-trạng thái kết thúc.
ACTIONS = ("GoStore", "GoHome", "PickUp", "Deliver", "Wait")

# Hệ số chiết khấu cho các reward trong tương lai.
GAMMA = 0.90

# Các action di chuyển thành công trong phần lớn trường hợp.
MOVE_SUCCESS_PROBABILITY = 0.90

# Giới hạn số bước giúp tránh episode vô hạn khi policy chưa tốt.
MAX_STEPS = 15

# Liệt kê toàn bộ không gian trạng thái.
STATES = tuple((location, status) for location in LOCATIONS for status in PACKAGE_STATUSES)

# Việc học và điều khiển chỉ cần quyết định ở các non-trạng thái kết thúc.
NONTERMINAL_STATES = tuple(state for state in STATES if state[1] != "delivered")


def is_terminal(state):
    """Trả về True khi gói hàng đã được giao."""
    # Thành phần thứ hai của state lưu trạng thái gói hàng.
    return state[1] == "delivered"


def reset_state(rng):
    """Lấy mẫu trạng thái khởi đầu cho một episode mới."""
    # Robot có thể bắt đầu ở một trong hai vị trí.
    return (rng.choice(LOCATIONS), "waiting")


def state_label(state):
    """Tạo nhãn dễ đọc cho tuple state."""
    # Tách state thành các biến có tên rõ ràng.
    location, status = state

    # Định dạng state để in trace và policy dễ đọc.
    return f"location={location}, package={status}"


print("States:", len(STATES))
print("Non-terminal states:", len(NONTERMINAL_STATES))
print("Actions:", ACTIONS)



States: 6
Non-terminal states: 4
Actions: ('GoStore', 'GoHome', 'PickUp', 'Deliver', 'Wait')


## Exercise 0 - Environment step function

Complete delivery_step. This function defines the environment transition $P(s' \mid s, a)$ implicitly through sampling, and returns the observed reward $r$.



In [18]:
def delivery_step(state, action, rng):
    """Lấy mẫu một transition trong môi trường giao hàng."""
    # Tách tuple state thành các biến có ý nghĩa.
    location, status = state

    # Khi đã giao hàng, episode kết thúc và không sinh thêm reward.
    if is_terminal(state):
        return state, 0.0

    # TODO: cài đặt từng action theo các luật ở trên.
    # Chỉ giữ tính ngẫu nhiên trong các action di chuyển.
    reward = 0.0
    if action == "GoStore":
        if rng.random() < MOVE_SUCCESS_PROBABILITY:
            location = "Store"
            
    elif action == "GoHome":
        if rng.random() < MOVE_SUCCESS_PROBABILITY:
            location = "Home"
            
    elif action == "PickUp":
        if location == "Store" and status == "waiting":
            status = "carrying"
            reward = 5.0  # PickUp thành công
        else:
            reward = -3.0  # Hành động không hợp lệ
            
    elif action == "Deliver":
        if location == "Home" and status == "carrying":
            status = "delivered"
            reward = 20.0  # Deliver thành công
        else:
            reward = -3.0  # Hành động không hợp lệ
            
    elif action == "Wait":
        pass
        
    else:
        raise ValueError(f"Unknown action: {action}")
    
    # TODO: trả về (next_state, reward).
    return (location, status), reward
    raise NotImplementedError("TODO: return the transition result")


In [19]:
# Kiểm tra Bài 0 sau khi hoàn thành TODO.
# Dùng RNG cố định để chuyển động ngẫu nhiên có thể tái lập khi cần.
rng = random.Random(0)

# PickUp tại Store khi gói hàng đang waiting phải hợp lệ và có reward tốt.
assert delivery_step(("Store", "waiting"), "PickUp", rng) == (("Store", "carrying"), 5.0)

# Deliver tại Home khi đang carrying phải đi đến trạng thái kết thúc.
assert delivery_step(("Home", "carrying"), "Deliver", rng) == (("Home", "delivered"), 20.0)

# Cố PickUp khi không ở Store phải là hành động không hợp lệ.
assert delivery_step(("Home", "waiting"), "PickUp", rng)[1] == -3.0

# Trạng thái kết thúc phải giữ nguyên và có reward bằng 0.
assert delivery_step(("Home", "delivered"), "Wait", rng) == (("Home", "delivered"), 0.0)

print("Exercise 0 checks passed.")

Exercise 0 checks passed.


## Common evaluation helpers

All three methods will use the same discounted return:

$
G_0 = \sum_{t=0}^{T-1} \gamma^t r_{t+1}
$



In [20]:
def run_episode(policy, start_state=("Store", "waiting"), seed=0, max_steps=MAX_STEPS, return_trace=False):
    """Chạy một episode và trả về return hoặc trace đầy đủ."""
    # Dùng RNG cục bộ để việc lấy mẫu môi trường có thể tái lập.
    rng = random.Random(seed)

    # Khởi tạo episode từ start_state được yêu cầu.
    state = start_state

    # Cộng dồn reward đã chiết khấu tại đây.
    total_return = 0.0

    # Reward đầu tiên có trọng số chiết khấu gamma^0 = 1.
    discount = 1.0

    # Lưu transition để có thể gỡ lỗi hoặc trực quan hóa.
    trace = []

    # Giữ episode hữu hạn ngay cả khi policy chưa tốt.
    for _ in range(max_steps):
        # Dừng khi việc giao hàng hoàn tất.
        if is_terminal(state):
            break

        # Hỏi policy cần chọn action nào tại state hiện tại.
        action = policy(state)

        # Lấy mẫu next_state và reward từ môi trường.
        next_state, reward = delivery_step(state, action, rng)

        # Lưu transition vừa quan sát.
        trace.append((state, action, reward, next_state))

        # Cộng reward đã chiết khấu vào return.
        total_return += discount * reward

        # Tăng lũy thừa của gamma cho reward tiếp theo.
        discount *= GAMMA

        # Cập nhật state hiện tại sang next_state.
        state = next_state

    # Chỉ trả về trace đầy đủ khi được yêu cầu.
    if return_trace:
        return total_return, trace

    # Nếu không, chỉ trả về đại lượng hiệu năng dạng số.
    return total_return


def evaluate_policy(policy, episodes=100, seed=100):
    """Ước lượng kỳ vọng discounted return của policy bằng mô phỏng."""
    # Dùng RNG có thể tái lập cho các trạng thái bắt đầu.
    rng = random.Random(seed)

    # Lưu một return cho mỗi episode.
    returns = []

    # Lấy trung bình trên nhiều trạng thái bắt đầu và kết quả ngẫu nhiên.
    for episode_index in range(episodes):
        # Lấy mẫu một trạng thái bắt đầu mới.
        start_state = reset_state(rng)

        # Dùng seed thay đổi cho mỗi episode mô phỏng.
        returns.append(run_episode(policy, start_state=start_state, seed=seed + episode_index))

    # Trả về trung bình mẫu.
    return statistics.mean(returns)


def print_policy(policy_table, title="Policy"):
    """In một action cho mỗi non-trạng thái kết thúc."""
    print(title)

    # Giữ thứ tự hiển thị cố định bằng cách duyệt qua NONTERMINAL_STATES.
    for state in NONTERMINAL_STATES:
        print(f"  {state_label(state):36s} -> {policy_table.get(state, '?')}")



# Part 1 - Adaptive Dynamic Programming (ADP)

ADP is model-based. You will learn an empirical model from experience:

$
\hat{P}(s' \mid s, a) = \frac{N(s, a, s')}{\sum_x N(s, a, x)}
$

and an empirical reward model:

$
\hat{R}(s, a, s') = \text{average observed reward for } (s, a, s')
$

Then you will solve the learned model with value iteration.



## Exercise 1.1 - Collect model data

Use random exploration to fill transition counts $N(s, a, s')$ and reward sums.



In [21]:
def collect_random_experience(episodes=300, seed=1):
    """Thu thập transition ngẫu nhiên để ước lượng model."""
    # Dùng RNG có thể tái lập cho cả chọn action và lấy mẫu môi trường.
    rng = random.Random(seed)

    # transition_counts[(state, action)][next_state] lưu N(s, a, s_next).
    transition_counts = defaultdict(Counter)

    # reward_sums[(state, action, next_state)] lưu tổng reward cho loại transition đó.
    reward_sums = defaultdict(float)

    # Thu thập nhiều episode ngắn.
    for episode_index in range(episodes):
        # Bắt đầu mỗi episode từ một trạng thái khởi đầu hợp lệ ngẫu nhiên.
        state = reset_state(rng)

        # Giới hạn độ dài của mỗi episode.
        for _ in range(MAX_STEPS):
            # Dừng thu thập nếu đã đạt trạng thái kết thúc.
            if is_terminal(state):
                break

            # TODO: chọn một action ngẫu nhiên, thực hiện một bước trong môi trường,
            # và cập nhật transition_counts cùng reward_sums.
            # Các key gợi ý:
            # transition_counts[(state, action)][next_state] += 1
            # reward_sums[(state, action, next_state)] += reward
            action = rng.choice(["GoStore", "GoHome", "PickUp", "Deliver", "Wait"])
            next_state, reward = delivery_step(state, action, rng)
            transition_counts[(state, action)][next_state] += 1
            reward_sums[(state, action, next_state)] += reward
            state = next_state

    # Trả về thống kê thô để bài tiếp theo ước lượng model.
    return transition_counts, reward_sums


# Thu thập dữ liệu bằng khám phá ngẫu nhiên.
transition_counts, reward_sums = collect_random_experience()

print("Observed state-action pairs:", len(transition_counts))



Observed state-action pairs: 20


In [22]:
# Kiểm tra Bài 1.1 sau khi hoàn thành TODO.
# Cần quan sát được ít nhất một cặp cặp trạng thái-hành động.
assert len(transition_counts) > 0

# Mọi cặp cặp trạng thái-hành động đã quan sát phải có transition count dương.
assert all(sum(counter.values()) > 0 for counter in transition_counts.values())

print("Exercise 1.1 checks passed.")

Exercise 1.1 checks passed.


## Exercise 1.2 - Build the learned model

Convert counts into probabilities with maximum likelihood estimation:

$
\hat{P}(s' \mid s, a) = \frac{N(s, a, s')}{N(s, a)}
$

where:

$
N(s, a) = \sum_x N(s, a, x)
$



In [24]:
def build_model(transition_counts, reward_sums):
    """Chuyển transition count và reward sum thành các model thực nghiệm."""
    # transition_model[(state, action)] sẽ ánh xạ next_state -> xác suất.
    transition_model = {}

    # reward_model[(state, action, next_state)] sẽ ánh xạ đến trung bình reward.
    reward_model = {}

    # Xử lý từng cặp cặp trạng thái-hành động đã quan sát.
    for state_action, next_counter in transition_counts.items():
        # TODO: chuyển count thành xác suất.
        total_count = sum(next_counter.values())

        # Tạo dictionary lồng nhau cho cặp cặp trạng thái-hành động này.
        transition_model[state_action] = {}

        # Ước lượng xác suất cho từng next_state đã quan sát.
        for next_state, count in next_counter.items():
            # TODO: chia count của next_state này cho total_count.
            probability = count / total_count

            # Lưu transition xác suất đã ước lượng.
            transition_model[state_action][next_state] = probability

            # TODO: tính trung bình reward cho transition này.
            reward_model[(state_action[0], state_action[1], next_state)] = reward_sums[(state_action[0], state_action[1], next_state)] / next_counter[next_state] if next_counter[next_state] > 0 else 0

    # Trả về cả hai thành phần của model đã học.
    return transition_model, reward_model


# Xây dựng transition model và reward model thực nghiệm từ dữ liệu đã thu thập.
transition_model, reward_model = build_model(transition_counts, reward_sums)

print("Learned transition-model entries:", len(transition_model))

Learned transition-model entries: 20


In [25]:
# Kiểm tra Bài 1.2 sau khi hoàn thành TODO.
# Model đã học phải có cùng các key cặp trạng thái-hành động như bảng count.
assert len(transition_model) == len(transition_counts)

# Mỗi phân phối transition phải có tổng bằng 1.
for state_action, distribution in transition_model.items():
    assert abs(sum(distribution.values()) - 1.0) < 1e-12

print("Exercise 1.2 checks passed.")

Exercise 1.2 checks passed.


## Exercise 1.3 - Plan with value iteration

For each non-terminal state, value iteration uses the Bellman optimality backup under the learned model:

$
V(s) = \max_a \sum_{s'} \hat{P}(s' \mid s, a) \left[ \hat{R}(s, a, s') + \gamma V(s') \right]
$



In [26]:
def model_q_value(state, action, values, transition_model, reward_model, gamma=GAMMA):
    """Tính Q(s, a) bằng transition model và reward model đã học."""
    # Nếu một action chưa từng được quan sát, phạt nó nhưng vẫn giữ phép tính xác định.
    if (state, action) not in transition_model:
        return -5.0 + gamma * values[state]

    # Cộng dồn kỳ vọng return trên các next_state có thể xảy ra.
    total = 0.0

    # TODO: tính action utility kỳ vọng từ model đã học.
    for next_state, probability in transition_model[(state, action)].items():
        reward = reward_model.get((state, action, next_state), 0.0)
        total += probability * (reward + gamma * values[next_state])

    # Trả về action utility theo model.
    return total


def value_iteration_from_model(transition_model, reward_model, iterations=80):
    """Dùng value iteration để tính tham lam policy từ model đã học."""
    # Khởi tạo value của mọi state bằng 0.
    values = {state: 0.0 for state in STATES}

    # Lặp các Bellman optimality backup.
    for _ in range(iterations):
        # Sao chép value cũ để mỗi lượt quét dùng ước lượng từ lượt trước.
        new_values = values.copy()

        # Chỉ cập nhật các non-trạng thái kết thúc.
        for state in NONTERMINAL_STATES:
            # TODO: thực hiện Bellman optimality backup bằng model_q_value.
            new_values[state] = max(model_q_value(state, action, values, transition_model, reward_model) for action in ACTIONS)

        # Chuyển sang bảng value đã cập nhật.
        values = new_values

    # Trích xuất tham lam policy từ bảng value cuối cùng.
    policy = {}

    # Chọn action tốt nhất trong từng non-trạng thái kết thúc.
    for state in NONTERMINAL_STATES:
        # TODO: chọn action có model_q_value lớn nhất.
        policy[state] = max(ACTIONS, key=lambda action: model_q_value(state, action, values, transition_model, reward_model))

    # Trả về cả bảng value và policy suy ra.
    return values, policy


# Lập kế hoạch bằng model đã học.
adp_values, adp_policy = value_iteration_from_model(transition_model, reward_model)

print_policy(adp_policy, "ADP policy")
print("ADP mean return:", evaluate_policy(lambda state: adp_policy.get(state, "Wait")))

ADP policy
  location=Store, package=waiting      -> PickUp
  location=Store, package=carrying     -> GoHome
  location=Home, package=waiting       -> GoStore
  location=Home, package=carrying      -> Deliver
ADP mean return: 19.943718000000004


In [27]:
# Kiểm tra Bài 1.3 sau khi hoàn thành các TODO.
# Policy phải chỉ định một action cho mỗi non-trạng thái kết thúc.
assert set(adp_policy.keys()) == set(NONTERMINAL_STATES)

# Mọi action được chọn phải hợp lệ.
assert all(action in ACTIONS for action in adp_policy.values())

# Một policy học được hợp lý nên đạt return dương trong bài toán này.
assert evaluate_policy(lambda state: adp_policy.get(state, "Wait"), episodes=30) > 0.0

print("Exercise 1.3 checks passed.")

Exercise 1.3 checks passed.


# Part 2 - Action-Utility Learning with Q-learning

Q-learning is model-free. It learns $Q(s, a)$ directly from transitions.

The update is:

$
Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]
$



In [29]:
def epsilon_greedy_action(q_values, state, epsilon, rng):
    """Chọn action bằng chiến lược epsilon-tham lam."""
    # TODO: khám phá ngẫu nhiên với xác suất epsilon;
    # ngược lại chọn action có Q-value cao nhất.
    if rng.random() < epsilon:
        return rng.choice(ACTIONS)
    else:
        return max(ACTIONS, key=lambda action: q_values[(state, action)])


def q_learning(episodes=800, alpha=0.25, seed=2):
    """Học bảng action-utility bằng Q-learning."""
    # Dùng RNG có thể tái lập cho action, trạng thái bắt đầu và transition.
    rng = random.Random(seed)

    # Các Q-value chưa biết được khởi tạo bằng 0.
    q_values = defaultdict(float)

    # Lưu các mốc kiểm tra hiệu năng định kỳ.
    scores = []

    # Chạy nhiều episode để học.
    for episode_index in range(episodes):
        # Bắt đầu mỗi episode từ một vị trí ngẫu nhiên khi gói hàng đang waiting.
        state = reset_state(rng)

        # Giảm khám phá theo thời gian nhưng vẫn giữ một lượng nhỏ khám phá.
        epsilon = max(0.05, 0.50 * (1 - episode_index / episodes))

        # Tương tác với môi trường trong một episode.
        for _ in range(MAX_STEPS):
            # Dừng khi gói hàng đã được giao.
            if is_terminal(state):
                break

            # Chọn action theo khám phá hoặc tham lam.
            action = epsilon_greedy_action(q_values, state, epsilon, rng)

            # Quan sát phản hồi của môi trường.
            next_state, reward = delivery_step(state, action, rng)

            # TODO: tính target của Q-learning và cập nhật q_values[(state, action)].
            target = reward + GAMMA * max(q_values[(next_state, a)] for a in ACTIONS)
            q_values[(state, action)] += alpha * (target - q_values[(state, action)])

            # Tiếp tục từ next_state.
            state = next_state

        # Đánh giá tham lam policy hiện tại tại các mốc đã chọn.
        if episode_index in {0, 9, 49, 199, episodes - 1}:
            # Chuyển Q-values thành tất định tham lam policy.
            policy = {state: max(ACTIONS, key=lambda action: q_values[(state, action)]) for state in NONTERMINAL_STATES}

            # Lưu số episode tại mốc kiểm tra và score đã ước lượng.
            scores.append((episode_index + 1, evaluate_policy(lambda s, p=policy: p.get(s, "Wait"), episodes=30, seed=300)))

    # Trích xuất tham lam policy cuối cùng từ Q.
    policy = {state: max(ACTIONS, key=lambda action: q_values[(state, action)]) for state in NONTERMINAL_STATES}

    # Trả về action utilities đã học, policy cuối cùng và các mốc kiểm tra tiến trình.
    return q_values, policy, scores


# Huấn luyện agent Q-learning.
q_values, q_policy, q_scores = q_learning()

print("Q-learning checkpoints:")
for episode, score in q_scores:
    print(f"  after {episode:>3} episodes: mean return = {score:+.3f}")

print_policy(q_policy, "Q-learning policy")
print("Q-learning mean return:", evaluate_policy(lambda state: q_policy.get(state, "Wait")))

Q-learning checkpoints:
  after   1 episodes: mean return = +4.767
  after  10 episodes: mean return = +20.108
  after  50 episodes: mean return = +20.108
  after 200 episodes: mean return = +20.108
  after 800 episodes: mean return = +20.108
Q-learning policy
  location=Store, package=waiting      -> PickUp
  location=Store, package=carrying     -> GoHome
  location=Home, package=waiting       -> GoStore
  location=Home, package=carrying      -> Deliver
Q-learning mean return: 19.943718000000004


In [30]:
# Kiểm tra Phần 2 sau khi hoàn thành các TODO.
# Policy đã học phải bao phủ mọi trạng thái quyết định.
assert set(q_policy.keys()) == set(NONTERMINAL_STATES)

# Mỗi action trong policy phải hợp lệ.
assert all(action in ACTIONS for action in q_policy.values())

# Một policy Q-learning hợp lý nên đạt return dương.
assert evaluate_policy(lambda state: q_policy.get(state, "Wait"), episodes=30) > 0.0

print("Part 2 checks passed.")

Part 2 checks passed.


# Part 3 - Policy Search

Policy search directly searches over policies. Because this environment has only four non-terminal states and five actions, a deterministic table policy has:

$
|A|^{|S_{nonterminal}|} = 5^4 = 625
$

possible policies.

This exhaustive search is small enough for practice, but the idea does not scale to large environments without smarter search methods.



In [31]:
def policy_from_action_tuple(action_tuple):
    """Chuyển action tuple thành tất định table policy."""
    # TODO: ánh xạ mỗi non-trạng thái kết thúc với một action trong action_tuple.
    state_action_pairs = zip(NONTERMINAL_STATES, action_tuple)
    return {state: action for state, action in state_action_pairs}


def exhaustive_policy_search():
    """Đánh giá mọi tất định table policy và giữ policy tốt nhất."""
    # Chưa có policy nào được chọn.
    best_policy = None

    # Khởi tạo best_score thấp hơn mọi score thực tế.
    best_score = float("-inf")

    # Đếm số policy đã được đánh giá.
    checked = 0

    # Liệt kê mọi cách gán action cho các non-trạng thái kết thúc.
    for action_tuple in product(ACTIONS, repeat=len(NONTERMINAL_STATES)):
        # Đếm candidate policy hiện tại.
        checked += 1

        # Chuyển tuple thành policy dạng dictionary.
        policy_table = policy_from_action_tuple(action_tuple)

        # TODO: đánh giá policy_table và giữ lại policy tốt nhất.
        score = evaluate_policy(lambda state: policy_table.get(state, "Wait"), episodes=30, seed=400)
        if score > best_score:
            best_score = score
            best_policy = policy_table

    # Trả về policy table tốt nhất tìm được và số candidate đã kiểm tra.
    return best_policy, best_score, checked


# Tìm kiếm trực tiếp trên các tất định policy.
search_policy, search_score, checked = exhaustive_policy_search()

print("Policies checked:", checked)
print_policy(search_policy, "Best policy found by policy search")
print("Policy-search mean return:", search_score)



Policies checked: 625
Best policy found by policy search
  location=Store, package=waiting      -> PickUp
  location=Store, package=carrying     -> GoHome
  location=Home, package=waiting       -> GoStore
  location=Home, package=carrying      -> Deliver
Policy-search mean return: 20.027800000000003


In [32]:
# Kiểm tra Phần 3 sau khi hoàn thành các TODO.
# Tìm kiếm vét cạn phải thử mọi tất định policy.
assert checked == len(ACTIONS) ** len(NONTERMINAL_STATES)

# Policy tốt nhất phải bao phủ tất cả non-trạng thái kết thúc.
assert set(search_policy.keys()) == set(NONTERMINAL_STATES)

# Mọi action được chọn phải hợp lệ.
assert all(action in ACTIONS for action in search_policy.values())

# Policy tốt nhất nên tạo ra return dương.
assert search_score > 0.0

print("Part 3 checks passed.")

Part 3 checks passed.


## Comparison and reflection

After all three parts run, compare the policies and answer these questions:

1. Which method needed a learned transition model $\hat{P}(s' \mid s, a)$?
2. Which method learned action utilities $Q(s, a)$ directly?
3. Which method searched directly in policy space?
4. Did all three methods find a reasonable delivery strategy? If not, which states are wrong and why?
5. How would the difficulty change if the environment had $100$ states instead of $6$?



In [33]:
# Cell so sánh tùy chọn: chạy sau khi hoàn thành tất cả các phần.
# Đặt mỗi bộ điều khiển đã học sau cùng một giao diện policy.
methods = {
    "ADP": lambda state: adp_policy.get(state, "Wait"),
    "Q-learning": lambda state: q_policy.get(state, "Wait"),
    "Policy search": lambda state: search_policy.get(state, "Wait"),
}

# So sánh mean return ước lượng bằng cùng một evaluation seed.
for name, policy in methods.items():
    print(f"{name:>13}: {evaluate_policy(policy, episodes=200, seed=999):+.3f}")

print("\nSample traces from (Store, waiting):")

# In một trace đại diện cho mỗi phương pháp.
for name, policy in methods.items():
    # Chạy một trace từ trạng thái bắt đầu cố định.
    total, trace = run_episode(policy, start_state=("Store", "waiting"), seed=42, return_trace=True)

    # Hiển thị tên phương pháp và return trước.
    print(f"\n{name}: return={total:+.3f}")

    # Hiển thị từng transition trong trace.
    for state, action, reward, next_state in trace:
        print(f"  {state_label(state):36s} --{action:>7}/{reward:+.1f}--> {state_label(next_state)}")

          ADP: +19.918
   Q-learning: +19.918
Policy search: +19.918

Sample traces from (Store, waiting):

ADP: return=+21.200
  location=Store, package=waiting      -- PickUp/+5.0--> location=Store, package=carrying
  location=Store, package=carrying     -- GoHome/+0.0--> location=Home, package=carrying
  location=Home, package=carrying      --Deliver/+20.0--> location=Home, package=delivered

Q-learning: return=+21.200
  location=Store, package=waiting      -- PickUp/+5.0--> location=Store, package=carrying
  location=Store, package=carrying     -- GoHome/+0.0--> location=Home, package=carrying
  location=Home, package=carrying      --Deliver/+20.0--> location=Home, package=delivered

Policy search: return=+21.200
  location=Store, package=waiting      -- PickUp/+5.0--> location=Store, package=carrying
  location=Store, package=carrying     -- GoHome/+0.0--> location=Home, package=carrying
  location=Home, package=carrying      --Deliver/+20.0--> location=Home, package=delivered
